### Import Required Dependencies

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import utils
importlib.reload(utils)
from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# --- Logging ---
logging.basicConfig(
    filename=log_dir / "project.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='w'
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

print("✅ Environment ready. Paths and logging configured.")



✅ Environment ready. Paths and logging configured.


# Section 2 Load and Process Dataset (national and Arkansas)

## Section 2A: Load utility functions

In [2]:
# Section 2A: Load utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,  # ✅ <- this line is key
    filter_to_county_level,
    log_duplicate_attributes,
    extract_common_year_from_columns,
    subset_columns_by_year,
    nca_counties
)

# Confirm it's working
logging.info("🧠 Utility functions from utils.py loaded successfully.")


[INFO] ✅ Utility functions loaded from utils.py
[INFO] 🧠 Utility functions from utils.py loaded successfully.


## Section 2B: Load and process national datasets

In [3]:
# Section 2B: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

complete_data = {}
state_lookup = None

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'fips_code': 'fips', 'fipstxt': 'fips', 'area_name': 'county'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()
        df['attribute'] = df['attribute'].str.strip().str.lower()

        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.strip().str.lower()
            state_lookup['state'] = state_lookup['state'].str.strip().str.upper()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"✅ Loaded and cleaned {filename}")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")

# Pivot and merge
edu_wide = complete_data['edu'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = complete_data['pop'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = complete_data['poverty'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = complete_data['unemp'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

df_full = reduce(lambda l, r: pd.merge(l, r, on='county', how='outer'),
                 [edu_wide, pop_wide, poverty_wide, unemp_wide])

df_full = df_full.reset_index().merge(state_lookup, on='county', how='left')
df_full['state'] = df_full['state'].str.upper()
df_full.set_index('county', inplace=True)

logging.info(f"✅ Final dataset shape: {df_full.shape}")
df_full.to_csv(output_dir / "us_county_merged.csv")


[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[INFO] ✅ Loaded and cleaned Education2023.csv
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[INFO] ✅ Loaded and cleaned PopulationEstimates.csv
[WARNING] ⚠️ poverty has duplicate county-attribute pairs.
[INFO] ✅ Loaded and cleaned Poverty2023.csv
[INFO] ✅ Loaded and cleaned Unemployment2023.csv
[INFO] ✅ Final dataset shape: (6497, 250)


## Section 2C: Subset Arkansas and NCA Counties

In [4]:
# Section 2C: Subset Arkansas and NCA counties

df_ar = df_full[df_full['state'] == 'AR'].copy()
df_ar = df_ar.reset_index()
df_ar['county'] = df_ar['county'].str.replace(" county", "", regex=False).str.replace(", ar", "", regex=False).str.strip()
df_ar.set_index('county', inplace=True)

df_nca = df_ar[df_ar.index.isin(nca_counties)].copy()
df_ar.to_csv(output_dir / "arkansas_counties.csv")
df_nca.to_csv(output_dir / "nca_counties.csv")

logging.info(f"📌 Arkansas counties: {df_ar.shape[0]}")
logging.info(f"📌 NCA counties: {df_nca.shape[0]}")


[INFO] 📌 Arkansas counties: 75
[INFO] 📌 NCA counties: 13


## Section 3: Exploratory Data Analysis (EDA)

In [5]:
# SECTION 3: Exploratory Data Analysis (EDA)

# Use a working copy of the NCA subset
df = df_nca.copy()
logging.info(f"🔍 Starting EDA on NCA dataset: {df.shape[0]} counties, {df.shape[1]} features")


[INFO] 🔍 Starting EDA on NCA dataset: 13 counties, 250 features


### Section 3.1: Inspect Dataset

In [6]:
# Inspect structure
display(df.info())
display(df.describe(include='all',))
display(df.head())


<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, baxter to woodruff
Columns: 250 entries, 2013 rural-urban continuum code to state
dtypes: float64(249), object(1)
memory usage: 25.5+ KB


None

,2013 rural-urban continuum code,2013 urban influence code,2023 rural-urban continuum code,2024 urban influence code,"bachelor's degree or higher, 1990","bachelor's degree or higher, 2000","bachelor's degree or higher, 2008-12","bachelor's degree or higher, 2019-23","four years of college or higher, 1970","four years of college or higher, 1980",...,unemployment_rate_2016,unemployment_rate_2017,unemployment_rate_2018,unemployment_rate_2019,unemployment_rate_2020,unemployment_rate_2021,unemployment_rate_2022,unemployment_rate_2023,urban_influence_code_2013_y,state
count,13.000000,13.00000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
mean,7.615385,8.00000,7.923077,7.307692,1310.000000,2006.461538,2476.769231,3122.230769,391.000000,994.538462,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.502135,2.27303,1.552500,2.015962,1037.041626,1797.723496,2403.215768,2742.226989,314.967459,723.642132,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,4.000000,4.00000,4.000000,3.000000,373.000000,460.000000,540.000000,693.000000,90.000000,347.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,7.000000,6.00000,7.000000,6.000000,529.000000,864.000000,964.000000,1408.000000,162.000000,431.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,8.000000,9.00000,9.000000,8.000000,876.000000,1261.000000,1394.000000,1967.000000,282.000000,646.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,9.000000,10.00000,9.000000,9.000000,2078.000000,3118.000000,3264.000000,4279.000000,599.000000,1480.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,2013 rural-urban continuum code,2013 urban influence code,2023 rural-urban continuum code,2024 urban influence code,"bachelor's degree or higher, 1990","bachelor's degree or higher, 2000","bachelor's degree or higher, 2008-12","bachelor's degree or higher, 2019-23","four years of college or higher, 1970","four years of college or higher, 1980",...,unemployment_rate_2016,unemployment_rate_2017,unemployment_rate_2018,unemployment_rate_2019,unemployment_rate_2020,unemployment_rate_2021,unemployment_rate_2022,unemployment_rate_2023,urban_influence_code_2013_y,state
county,,,,,,,,,,,,,,,,,,,,,
baxter,7.0,8.0,7.0,7.0,2436.0,3688.0,4873.0,6029.0,599.0,1761.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR
cleburne,8.0,4.0,8.0,3.0,529.0,879.0,964.0,1978.0,162.0,431.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR
fulton,9.0,9.0,9.0,9.0,373.0,864.0,922.0,1408.0,173.0,347.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR
independence,7.0,8.0,7.0,7.0,2078.0,3118.0,3264.0,4279.0,620.0,1480.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR
izard,9.0,10.0,9.0,9.0,771.0,1112.0,1162.0,1967.0,158.0,646.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR


### Section 3.2 Extract and Rename Key Variables

In [7]:
## Section 3.2: Extract and Name Key Indicator Variables

# Use the utility function to extract standardized key indicators from df_nca
df_nca, used_columns, year = extract_key_indicators(df_nca)

# Print summary of what was extracted
print("📅 Most common year in column names:", year)
print("📚 Education columns used:", used_columns['education'])
print("📉 Poverty columns used:", used_columns['poverty'])
print("💼 Unemployment columns used:", used_columns['unemployment'])
print("👥 Population columns used:", used_columns['population'])

# Define the standardized variable names for analysis
variables = ['BachelorsDegreeRate', 'HighSchoolGradRate', 'PovertyRate', 'UnemploymentRate', 'Population']

# Optional: Display a preview
df_nca[variables].head()


[INFO] 📅 Most common year in column names: 2023
[INFO] 📚 Education columns: []
[INFO] 📉 Poverty columns: []
[INFO] 💼 Unemployment columns: ['unemployment_rate_2023']
[INFO] 👥 Population columns: []


📅 Most common year in column names: 2023
📚 Education columns used: []
📉 Poverty columns used: []
💼 Unemployment columns used: ['unemployment_rate_2023']
👥 Population columns used: []


,BachelorsDegreeRate,HighSchoolGradRate,PovertyRate,UnemploymentRate,Population
county,,,,,
baxter,NaN,NaN,NaN,NaN,NaN
cleburne,NaN,NaN,NaN,NaN,NaN
fulton,NaN,NaN,NaN,NaN,NaN
independence,NaN,NaN,NaN,NaN,NaN
izard,NaN,NaN,NaN,NaN,NaN


### Section 3.3 Visualize Education Levels

In [8]:
# Education distribution across counties
title = "Education Indicators by County"
df[education_cols].T.plot(kind='bar', figsize=(14, 6), title=title)
plt.ylabel("Percent or Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Standardize filename
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f" Saved: {filename}.png")

plt.show()



NameError: name 'education_cols' is not defined

### Section 3.4: Distribution Plots (Histograms & KDE)

In [ ]:
# Distribution plots with title-based saving
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate']

for var in variables:
    plt.figure(figsize=(8, 4))
    
    # Define plot title
    title = f"Distribution of {var}"
    
    # Plot
    sns.histplot(df[var], kde=True, bins=20)
    plt.title(title)
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    
    # Create safe filename from title
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")
    
    plt.show()


### Section 3.5: Correlation Heatmap

In [ ]:
# Correlation matrix and heatmap
corr_vars = df[['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate', 'Population']]
corr_matrix = corr_vars.corr()

# Define title
title = "Correlation Between Key Indicators"

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title(title)
plt.tight_layout()

# Generate safe filename from title
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")

plt.show()


### Section 3.6: Key Relationships (Scatter Plots)

In [ ]:
# Scatter Plots of Key Relationships
# 1. Bachelor's Degree vs Poverty
title = "Bachelor's Degree Rate vs. Poverty Rate"
sns.scatterplot(x='BachelorsDegreeRate', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Bachelor's Degree (%)")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 2. High School Grad Rate vs Unemployment
title = "High School Grad Rate vs. Unemployment Rate"
sns.scatterplot(x='HighSchoolGradRate', y='UnemploymentRate', data=df)
plt.title(title)
plt.xlabel("High School Grad (%)")
plt.ylabel("Unemployment Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 3. Population vs Poverty Rate
title = "Population vs. Poverty Rate"
sns.scatterplot(x='Population', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Population")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()


### Section 3.7: Outlier Detection with BoxPlots

In [ ]:
# --- Boxplots for Outlier Detection (with title-based save) ---
for var in variables:
    plt.figure(figsize=(8, 4))

    # Define title and filename
    title = f"Boxplot of {var}"
    sns.boxplot(x=df[var])
    plt.title(title)
    plt.tight_layout()

    # Standardize filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")

    plt.show()


### Section 3.8 ParPlot for Key Indicators

In [ ]:
# Pairplot for Key Indicators
sns.pairplot(df[variables], diag_kind='kde')
plt.suptitle("Pairwise Relationships Between Key Indicators", y=1.02)
plt.tight_layout()
plt.savefig(image_dir / "pairplot_key_indicators.png", dpi=300, bbox_inches='tight')
logging.info("📷 Saved: pairplot_key_indicators.png")
plt.show()


### Step 4: Visualize Distributions (Histograms & KDE)

### Step 5: Correlation Matrix and Heatmap

### Step 6: Scatter Plots for Key Relationships

### Step 7: Identify Outlier Counties with Boxplots

### Step 8: Log-Transform Population (Optional)
If the population is skewed: